In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import ( col, when, trim, lit, regexp_replace, split, explode, lower, array_contains, transform, isnan, count, round)
from pyspark.sql.types import IntegerType
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Book_Movie_Adapt-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com sucesso e Delta Lake configurado.")

SparkSession iniciada com sucesso e Delta Lake configurado.


In [24]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/book_movie_adapt.json"

In [25]:
df_bronze = spark.read.json(hdfs_path)

In [26]:
print(f"Total de registos lidos da camada Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.toPandas()

Total de registos lidos da camada Bronze: 18111
root
 |-- authorLabel: string (nullable = true)
 |-- bookLabel: string (nullable = true)
 |-- book_year: string (nullable = true)
 |-- filmLabel: string (nullable = true)
 |-- release_year: string (nullable = true)



,authorLabel,bookLabel,book_year,filmLabel,release_year
0,Brothers Grimm,Snow White,1812,Snow White,2025
1,Brothers Grimm,Snow White,1812,The Death of Snow White,2025
2,Charles Dickens,A Christmas Carol,1843,Ta Kalanta ton Hristougennon,2025
3,Homer,Odyssey,None,The Return,2025
4,None,Lilo & Stitch,2002,Lilo & Stitch,2025
...,...,...,...,...,...
18106,Aleksis Kivi,The Engagement,1866,Q11871325,1920
18107,Adolph L'Arronge,Hasemann's Daughters,1879,Hasemann's Daughters,1920
18108,Anni Swan,Q23045569,1919,Olli’s Apprenticeship,1920
18109,Herman Heijermans,Schakels,1903,Schakels,1920


In [27]:
from pyspark.sql.functions import col, count

print(f"Total de registos no 'df_bronze': {df_bronze.count()}")

# Procura por linhas onde 'book_year' é inválido
book_year_invalidos = df_bronze.filter(col("book_year") <= 0)

print(f"Total de 'book_year' inválidos (<= 0): {book_year_invalidos.count()}")

book_year_invalidos.select("filmLabel", "bookLabel", "book_year").orderBy(col("book_year")).show(5)

Total de registos no 'df_bronze': 18111
Total de 'book_year' inválidos (<= 0): 2
+--------------------+--------------------+---------+
|           filmLabel|           bookLabel|book_year|
+--------------------+--------------------+---------+
|Caesar the Conqueror|Commentarii de Be...|      -48|
|Caesar the Conqueror|Commentarii de Be...|      -57|
+--------------------+--------------------+---------+



In [28]:
from pyspark.sql.functions import when, lit

# Cria o novo DataFrame 'df_book_year' com a coluna corrigida
df_book_year = df_bronze.withColumn("book_year",
    when(
        col("book_year") <= 0,
        lit(None)  # A forma de inserir um valor NULO
    ).otherwise(col("book_year")) # Mantém o valor original se for > 0
)

In [29]:
invalidos_depois = df_book_year.filter(col("book_year") <= 0).count()

if invalidos_depois == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'book_year' inválidos.")
else:
    print(f"❌ FALHA: Ainda existem {invalidos_depois} valores inválidos.")

nulos_book_year = df_book_year.filter(col("book_year").isNull()).count()

print(f"\nTotal de registos com 'book_year' = NULL: {nulos_book_year}")

✅ SUCESSO: Não foram encontrados mais 'book_year' inválidos.

Total de registos com 'book_year' = NULL: 4572


In [30]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_book_year': {df_book_year.count()}")

# Procura por linhas onde 'authorLabel' é nulo ou uma string vazia (após trim)
author_vazios = df_book_year.filter(
    (col("authorLabel").isNull()) | (trim(col("authorLabel")) == "")
)

author_vazios.select("filmLabel", "bookLabel", "authorLabel").show(5)
print(f"Total de 'authorLabel' nulos/vazios encontrados: {author_vazios.count()}")

Total de registos no DataFrame 'df_book_year': 18111
+--------------------+--------------------+-----------+
|           filmLabel|           bookLabel|authorLabel|
+--------------------+--------------------+-----------+
|       Lilo & Stitch|       Lilo & Stitch|       null|
|     The Last Supper|         Last Supper|       null|
|   A Minecraft Movie|           Minecraft|       null|
|Downton Abbey: Th...|       Downton Abbey|       null|
|          Snow White|Snow White and th...|       null|
+--------------------+--------------------+-----------+
only showing top 5 rows

Total de 'authorLabel' nulos/vazios encontrados: 4251


In [31]:
from pyspark.sql.functions import lit

df_authorLabel = df_book_year.withColumn("authorLabel",
    when(
        (col("authorLabel").isNull()) | (trim(col("authorLabel")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("authorLabel")) # Mantém o valor original
)

In [32]:
author_vazios_depois = df_authorLabel.filter(
    (col("authorLabel").isNull()) | (trim(col("authorLabel")) == "")
)
num_vazias = author_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'authorLabel' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'authorLabel' nulos/vazios.")

author_na = df_authorLabel.filter(col("authorLabel") == "NA")
print(f"Total de registos com authorLabel = 'NA': {author_na.count()}")

✅ SUCESSO: Não foram encontrados mais 'authorLabel' nulos/vazios.
Total de registos com authorLabel = 'NA': 4251


In [33]:
from pyspark.sql.types import IntegerType

df_inteiro = df_authorLabel.withColumn(
    "book_year",
    col("book_year").cast(IntegerType())
).withColumn(
    "release_year",
    col("release_year").cast(IntegerType())
)

In [34]:
from pyspark.sql.functions import lower, trim, col, translate, regexp_replace

# Colunas de texto a limpar
text_columns_to_clean = ["authorLabel", "bookLabel", "filmLabel"]

# O nosso DataFrame de trabalho
df_final_limpo = df_inteiro

# Mapa de acentos
accent_map = {
    'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
    'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'ô': 'o', 'ç': 'c',
    'ñ': 'n', 'ü': 'u', 'ë': 'e', 'ï': 'i', 'ö': 'o',
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
    'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Ô': 'O', 'Ç': 'C',
    'Ñ': 'N', 'Ü': 'U', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O'
}

# Converte o mapa em duas strings para a função translate()
# Esta é a forma mais eficiente de o fazer em Spark
from_chars = "".join(accent_map.keys())
to_chars = "".join(accent_map.values())


for col_name in text_columns_to_clean:

# Começa com a coluna original
    cleaned_col = col(col_name)

    # Remove espaços no início/fim (Trim)
    cleaned_col = trim(cleaned_col)

    # Remove acentos (Diacritic Folding)
    # Usamos translate() que é muito mais rápido que múltiplos regexp_replace
    cleaned_col = translate(cleaned_col, from_chars, to_chars)

    # Converte para minúsculo (Lowercase)
    cleaned_col = lower(cleaned_col)

    # Remover espaços duplos
    cleaned_col = regexp_replace(cleaned_col, " +", " ")

    # Atualiza o DataFrame com a coluna limpa
    df_final_limpo = df_final_limpo.withColumn(col_name, cleaned_col)

# Mostra o resultado
df_final_limpo.select(text_columns_to_clean).show(truncate=False)

+-------------------+---------------------------------------+-------------------------------+
|authorLabel        |bookLabel                              |filmLabel                      |
+-------------------+---------------------------------------+-------------------------------+
|brothers grimm     |snow white                             |snow white                     |
|brothers grimm     |snow white                             |the death of snow white        |
|charles dickens    |a christmas carol                      |ta kalanta ton hristougennon   |
|homer              |odyssey                                |the return                     |
|na                 |lilo & stitch                          |lilo & stitch                  |
|bram stoker        |dracula                                |nosferatu                      |
|bram stoker        |dracula                                |dracula: a love tale           |
|na                 |last supper                            

In [35]:
df_final_limpo.printSchema()

root
 |-- authorLabel: string (nullable = true)
 |-- bookLabel: string (nullable = true)
 |-- book_year: integer (nullable = true)
 |-- filmLabel: string (nullable = true)
 |-- release_year: integer (nullable = true)



In [36]:
# Salvando os dados normalizados na camada Silver (Delta ou Parquet)
df_final_limpo \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/book_movie_adapt")

In [37]:
df_final_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("release_year") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/book_movie_adapt") \
    .saveAsTable("projeto.book_movie_adapt")

In [38]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.book_movie_adapt
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      2|2025-11-17 17:15:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          1|  Serializable|        false|{numFiles -> 106,...|        null|Apache-Spark/3.4....|
|      1|2025-11-17 17:14:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -

In [39]:
spark.sql(
    """
    SELECT * FROM projeto.book_movie_adapt
    """
).show()

+--------------------+--------------------+---------+--------------------+------------+
|         authorLabel|           bookLabel|book_year|           filmLabel|release_year|
+--------------------+--------------------+---------+--------------------+------------+
|  arthur conan doyle|the hound of the ...|     1902|der hund von bask...|        1929|
|     alexandre dumas|the count of mont...|     1844|the count of mont...|        1929|
| william shakespeare|the taming of the...|     1623|the taming of the...|        1929|
|         jules verne|the mysterious is...|     1874|the mysterious is...|        1929|
|              q35064|the secret adversary|     1922|     adventures inc.|        1929|
|  frederick lonsdale|the last of mrs. ...|     1925|the last of mrs. ...|        1929|
|  arthur conan doyle|the return of she...|     1905|the return of she...|        1929|
|     alexandre dumas|the vicomte of br...|     1847|       the iron mask|        1929|
|        stefan zweig|letter fro

In [40]:
spark.stop()